# Glotty evaluator: LoRA fine-tune of Qwen2.5-Coder-32B-Instruct

Trains a LoRA adapter for the Glotty evaluator on Kaggle (**Accelerator: GPU T4 x2**, **Internet: on**) and writes a Cloudflare-ready adapter.

**Inputs.** Upload `data/train/train.jsonl`, `val.jsonl` and `eval-anchor-dev.jsonl` from the repo as a Kaggle Dataset named `glotty-train` and add it to this notebook (Add Input). Built by `npm run train:export`, `train:codes`, `train:build`.

**Output.** `/kaggle/working/adapter/` with `adapter_config.json` and `adapter_model.safetensors`: download them and upload with `npx wrangler ai finetune create @cf/qwen/qwen2.5-coder-32b-instruct glotty-evaluator-v1 ./adapter`.

Things to check on the first run are marked **CHECK**.

In [ ]:
# Versions: recent transformers/peft/bitsandbytes. CHECK if an import fails: pin to the versions Kaggle's image supports.
!pip install -q -U "transformers>=4.46" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=1.0" datasets

In [ ]:
import json, os, math, time, torch
os.environ.setdefault("HF_HOME", "/tmp/hf")  # large scratch disk; /kaggle/working is limited to ~20 GB
DATA = "/kaggle/input/glotty-train"
OUT = "/kaggle/working"
# CHECK: a ready-made 4-bit copy of the base (~19 GB). The original fp16 weights are ~65 GB.
BASE = "unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit"
LORA_R, LORA_ALPHA = 8, 16          # rank 8 on all linear layers ~= 135 MB; Cloudflare allows rank <= 32 and <= 300 MB
EPOCHS, LR, GRAD_ACCUM, MAX_LEN = 1, 2e-4, 8, 1024
print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto", max_memory={0: "14GiB", 1: "14GiB"}, torch_dtype=torch.float16,
)
print(model.hf_device_map if hasattr(model, "hf_device_map") else "")

In [ ]:
def load(name):
    with open(f"{DATA}/{name}") as f:
        return [json.loads(l) for l in f if l.strip()]
train_rows, val_rows = load("train.jsonl"), load("val.jsonl")

def encode(row):
    # Loss on the assistant answer only: the prompt tokens are masked with -100.
    prefix = tok.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
    answer = row["messages"][-1]["content"] + tok.eos_token
    p = tok(prefix, add_special_tokens=False)["input_ids"]
    a = tok(answer, add_special_tokens=False)["input_ids"]
    ids = (p + a)[:MAX_LEN]
    return {"input_ids": ids, "labels": ([-100] * len(p) + a)[:MAX_LEN]}

train_ds = [encode(r) for r in train_rows]
val_ds = [encode(r) for r in val_rows]
lens = sorted(len(x["input_ids"]) for x in train_ds)
print(len(train_ds), "train,", len(val_ds), "val; tokens p50/p95/max:", lens[len(lens)//2], lens[int(len(lens)*.95)], lens[-1])

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments
def collate(batch):
    n = max(len(x["input_ids"]) for x in batch)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    return {
        "input_ids": torch.tensor([x["input_ids"] + [pad] * (n - len(x["input_ids"])) for x in batch]),
        "labels": torch.tensor([x["labels"] + [-100] * (n - len(x["labels"])) for x in batch]),
        "attention_mask": torch.tensor([[1] * len(x["input_ids"]) + [0] * (n - len(x["input_ids"])) for x in batch]),
    }
args = TrainingArguments(
    output_dir=f"{OUT}/checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
    lr_scheduler_type="cosine", warmup_ratio=0.05, fp16=True, optim="paged_adamw_8bit",
    logging_steps=10, save_steps=50, save_total_limit=2, report_to="none",
    gradient_checkpointing=True, remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
# If the session restarted, resume from the last checkpoint in /kaggle/working/checkpoints.
last = None
if os.path.isdir(args.output_dir):
    ck = sorted((d for d in os.listdir(args.output_dir) if d.startswith("checkpoint-")), key=lambda d: int(d.split("-")[1]))
    last = os.path.join(args.output_dir, ck[-1]) if ck else None
t0 = time.time()
trainer.train(resume_from_checkpoint=last)
print(f"trained in {(time.time()-t0)/3600:.1f} h")
print(trainer.evaluate())

In [ ]:
# Save the adapter in the shape Cloudflare expects: adapter_config.json + adapter_model.safetensors.
ADAPTER = f"{OUT}/adapter"
model.save_pretrained(ADAPTER, safe_serialization=True)
cfg_path = f"{ADAPTER}/adapter_config.json"
cfg = json.load(open(cfg_path))
cfg["model_type"] = "qwen2"   # CHECK against the Cloudflare LoRA docs for qwen2.5-coder-32b-instruct
cfg["base_model_name_or_path"] = "Qwen/Qwen2.5-Coder-32B-Instruct"
json.dump(cfg, open(cfg_path, "w"), indent=2)
for f in os.listdir(ADAPTER):
    print(f, round(os.path.getsize(f"{ADAPTER}/{f}") / 1e6, 1), "MB")

## Optional: measure on the anchor screening set before uploading

Greedy decoding on the 118 sentences of `anchor-cowsl2h-dev` (instructor-labelled, never trained on). On two T4s a 32B model is slow (roughly 30-60 s per item), so set `LIMIT` to run a sample first.

In [ ]:
LIMIT = 40
rows = load("eval-anchor-dev.jsonl")[:LIMIT]
model.eval(); tok.padding_side = "left"
def judge(messages):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=320, do_sample=False)
    reply = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        return json.loads(reply[reply.index("{"): reply.rindex("}") + 1])
    except Exception:
        return {"evaluation": "invalid"}
results = []
for i, r in enumerate(rows):
    v = judge(r["messages"])
    kind = "correct" if r["gold"]["evaluation"] == "correct" else ("por_para" if ("por ->" in r["gold"]["notes"] or "para ->" in r["gold"]["notes"]) else "aspect")
    results.append((kind, v.get("evaluation"), v.get("confidence")))
    if i % 10 == 9: print(i + 1, "done")
from collections import Counter
for k in ("por_para", "aspect", "correct"):
    print(k, Counter(e for kk, e, _ in results if kk == k))
pred_inc = [x for x in results if x[1] == "incorrect"]
print("precision on incorrect:", round(sum(x[0] != "correct" for x in pred_inc) / max(1, len(pred_inc)), 3))
json.dump(results, open(f"{OUT}/anchor-dev-results.json", "w"))